# Clinical Diabetes Dataset Cleaning & Standardization Pipeline (Palestine)

This notebook cleans, validates, and standardizes a real-world clinical diabetes dataset collected in Palestine (`data/Raw_Data.csv`) and exports the analysis-ready output (`data/NRaw_Data.csv`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

## 1. Load Raw Dataset & Initial Inspection

In [ ]:
# Load raw dataset
df = pd.read_csv('../data/Raw_Data.csv')

print(f"Dataset Shape: {df.shape}")
df.info()
df.head()

## 2. Categorical Standardization (`PatientSex` & `MaritalStatus`)

In [ ]:
# Clean whitespace and standardize casing
for col in ['PatientSex', 'MaritalStatus']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.title()

# Map common gender abbreviations/variants
if 'PatientSex' in df.columns:
    sex_map = {'M': 'Male', 'F': 'Female', 'Nan': np.nan, 'None': np.nan}
    df['PatientSex'] = df['PatientSex'].replace(sex_map)
    print("PatientSex value counts:\n", df['PatientSex'].value_counts(dropna=False))

# Standardize MaritalStatus variants
if 'MaritalStatus' in df.columns:
    marital_map = {'Nan': np.nan, 'None': np.nan, 'Unknown': np.nan}
    df['MaritalStatus'] = df['MaritalStatus'].replace(marital_map)
    print("\nMaritalStatus value counts:\n", df['MaritalStatus'].value_counts(dropna=False))

## 3. Numeric & String Parsing (`HBA1CResult` & `FastingBloodSugarResult`)

In [ ]:
# Clean HBA1CResult: strip '%' symbols and coerce malformed strings to NaN
if 'HBA1CResult' in df.columns:
    df['HBA1CResult'] = (
        df['HBA1CResult']
        .astype(str)
        .str.replace('%', '', regex=False)
        .str.strip()
    )
    df['HBA1CResult'] = pd.to_numeric(df['HBA1CResult'], errors='coerce')

# Clean FastingBloodSugarResult: coerce non-numeric strings/artifacts to NaN
if 'FastingBloodSugarResult' in df.columns:
    df['FastingBloodSugarResult'] = pd.to_numeric(
        df['FastingBloodSugarResult'].astype(str).str.strip(),
        errors='coerce'
    )

df[['HBA1CResult', 'FastingBloodSugarResult']].describe()

## 4. Data Integrity & Clinical Validation

In [ ]:
# Drop duplicate rows if any exist
df = df.drop_duplicates().copy()

# Remove records missing critical lab measurements
critical_cols = [c for c in ['HBA1CResult', 'FastingBloodSugarResult'] if c in df.columns]
df = df.dropna(subset=critical_cols).copy()

# Standardize clinical diagnostic flags based on standard medical thresholds
# HbA1c: Normal (< 5.7%), Prediabetes (5.7% - 6.4%), Diabetes (>= 6.5%)
if 'HBA1CResult' in df.columns:
    df['HBA1C_Category'] = pd.cut(
        df['HBA1CResult'],
        bins=[-np.inf, 5.69, 6.49, np.inf],
        labels=['Normal', 'Prediabetes', 'Diabetes']
    )

# Fasting Blood Sugar (mg/dL): Normal (< 100), Prediabetes (100 - 125), Diabetes (>= 126)
if 'FastingBloodSugarResult' in df.columns:
    df['FBS_Category'] = pd.cut(
        df['FastingBloodSugarResult'],
        bins=[-np.inf, 99.9, 125.9, np.inf],
        labels=['Normal', 'Prediabetes', 'Diabetes']
    )

print(f"Cleaned Dataset Shape: {df.shape}")
df.head()

## 5. Export Cleaned Dataset

In [ ]:
output_path = '../data/NRaw_Data.csv'
df.to_csv(output_path, index=False)
print(f"Cleaned dataset successfully saved to {output_path}")